# India Crypto Cashout Tax & Fee Calculator (2026)


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# India Crypto Cashout Tax & Fee Calculator (2026)

Based on India's crypto withdrawal framework covering 30% TDS, income tax slabs, and capital gains treatment. This notebook calculates net INR received after all taxes and fees when converting crypto to Indian Rupees.

**Key Assumptions:**
- 30% TDS applies to crypto sales (as per 2026 Indian tax code)
- Capital gains treated as income tax event
- Exchange fees vary by platform and withdrawal method
- Holding period affects tax treatment (short-term vs. long-term)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Define India tax brackets (2026 estimated, in INR)
TAX_BRACKETS = [
    (250000, 0.0),
    (500000, 0.05),
    (750000, 0.10),
    (1000000, 0.15),
    (1250000, 0.20),
    (1500000, 0.30),
    (float('inf'), 0.30)
]

TDS_RATE = 0.30  # 30% TDS on crypto transactions

def calculate_taxable_income_bracket(income):
    """Determine applicable tax bracket and rate."""
    for threshold, rate in TAX_BRACKETS:
        if income <= threshold:
            return rate
    return TAX_BRACKETS[-1][1]

def calculate_capital_gains_tax(sale_proceeds, cost_basis, holding_days):
    """Calculate capital gains tax based on holding period."""
    capital_gain = sale_proceeds - cost_basis
    
    if capital_gain <= 0:
        return 0, capital_gain, 'loss'
    
    # Long-term if held > 365 days (as per typical Indian rules)
    is_long_term = holding_days >= 365
    
    if is_long_term:
        # Long-term gains often taxed at lower indexation or rate
        gain_type = 'long-term'
        effective_rate = 0.20  # Assumed long-term rate
    else:
        gain_type = 'short-term'
        effective_rate = calculate_taxable_income_bracket(capital_gain)
    
    tax_on_gains = capital_gain * effective_rate
    return tax_on_gains, capital_gain, gain_type

def calculate_cashout(crypto_amount_usd, sale_price_usd, cost_basis_usd, 
                      holding_days, exchange_fee_percent, withdrawal_fee_usd, 
                      usd_to_inr_rate=83.5):
    """Calculate net INR received after all taxes and fees."""
    
    # Gross proceeds in USD
    gross_proceeds_usd = crypto_amount_usd * sale_price_usd
    
    # Apply exchange fee
    exchange_fee_usd = gross_proceeds_usd * (exchange_fee_percent / 100)
    proceeds_after_exchange_usd = gross_proceeds_usd - exchange_fee_usd
    
    # TDS calculation (30%)
    tds_amount_usd = gross_proceeds_usd * TDS_RATE
    
    # Capital gains tax
    cg_tax_usd, capital_gain_usd, gain_type = calculate_capital_gains_tax(
        gross_proceeds_usd, cost_basis_usd, holding_days
    )
    
    # Total tax (TDS + capital gains tax, not double-counted)
    total_tax_usd = tds_amount_usd + max(0, cg_tax_usd - tds_amount_usd)
    
    # Withdrawal fee
    net_proceeds_usd = proceeds_after_exchange_usd - withdrawal_fee_usd - total_tax_usd
    
    # Convert to INR
    net_proceeds_inr = net_proceeds_usd * usd_to_inr_rate
    
    return {
        'gross_proceeds_usd': gross_proceeds_usd,
        'exchange_fee_usd': exchange_fee_usd,
        'tds_usd': tds_amount_usd,
        'capital_gains_tax_usd': cg_tax_usd,
        'total_tax_usd': total_tax_usd,
        'withdrawal_fee_usd': withdrawal_fee_usd,
        'net_proceeds_usd': net_proceeds_usd,
        'net_proceeds_inr': net_proceeds_inr,
        'capital_gain_usd': capital_gain_usd,
        'gain_type': gain_type,
        'usd_to_inr_rate': usd_to_inr_rate
    }

print("India Crypto Cashout Calculator initialized.")

## Example Scenario: Cashout 2 BTC Held for 2 Years

Assumptions:
- Amount: 2 BTC
- Current price: $45,000 USD
- Cost basis: $30,000 USD (average purchase price)
- Holding period: 730 days (2 years, long-term)
- Exchange fee: 0.5%
- Bank withdrawal fee: $5
- USD to INR rate: 83.5

In [ ]:
# Example: Cashout 2 BTC
crypto_amount = 2  # BTC
sale_price = 45000  # USD per BTC
cost_basis_total = 60000  # Total cost basis in USD
holding_days = 730  # 2 years
exchange_fee = 0.5  # percent
withdrawal_fee = 5  # USD

result = calculate_cashout(
    crypto_amount_usd=crypto_amount * sale_price,
    sale_price_usd=1,  # Already converted to total
    cost_basis_usd=cost_basis_total,
    holding_days=holding_days,
    exchange_fee_percent=exchange_fee,
    withdrawal_fee_usd=withdrawal_fee,
    usd_to_inr_rate=83.5
)

print("\n=== CASHOUT SUMMARY ===")
print(f"Gross Proceeds: ${result['gross_proceeds_usd']:,.2f}")
print(f"Capital Gain: ${result['capital_gain_usd']:,.2f} ({result['gain_type']})")
print(f"Exchange Fee (0.5%): -${result['exchange_fee_usd']:,.2f}")
print(f"TDS (30%): -${result['tds_usd']:,.2f}")
print(f"Capital Gains Tax: -${result['capital_gains_tax_usd']:,.2f}")
print(f"Withdrawal Fee: -${result['withdrawal_fee_usd']:,.2f}")
print(f"\nNet Proceeds (USD): ${result['net_proceeds_usd']:,.2f}")
print(f"Net Proceeds (INR @ {result['usd_to_inr_rate']}): ₹{result['net_proceeds_inr']:,.2f}")
print(f"\nEffective Tax Rate: {(result['total_tax_usd'] / result['gross_proceeds_usd'] * 100):.1f}%")

## Scenario Comparison: Different Holding Periods

Let's compare net proceeds for the same transaction across different holding periods to show the impact of short-term vs. long-term gains treatment.

In [ ]:
# Compare scenarios with different holding periods
holding_periods = [30, 180, 365, 730, 1095]  # Days
scenarios = []

for days in holding_periods:
    result = calculate_cashout(
        crypto_amount_usd=90000,  # 2 BTC @ 45k
        sale_price_usd=1,
        cost_basis_usd=60000,
        holding_days=days,
        exchange_fee_percent=0.5,
        withdrawal_fee_usd=5,
        usd_to_inr_rate=83.5
    )
    scenarios.append({
        'Holding Period (Days)': days,
        'Gain Type': result['gain_type'],
        'Capital Gain (USD)': result['capital_gain_usd'],
        'Total Tax (USD)': result['total_tax_usd'],
        'Net INR': result['net_proceeds_inr']
    })

df_scenarios = pd.DataFrame(scenarios)
print("\n=== HOLDING PERIOD IMPACT ===")
print(df_scenarios.to_string(index=False))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(df_scenarios['Holding Period (Days)'].astype(str), df_scenarios['Total Tax (USD)'], color='#d62728')
ax1.set_ylabel('Total Tax (USD)', fontsize=11)
ax1.set_xlabel('Holding Period (Days)', fontsize=11)
ax1.set_title('Tax Liability by Holding Period', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

ax2.bar(df_scenarios['Holding Period (Days)'].astype(str), df_scenarios['Net INR']/100000, color='#2ca02c')
ax2.set_ylabel('Net INR (in hundreds of thousands)', fontsize=11)
ax2.set_xlabel('Holding Period (Days)', fontsize=11)
ax2.set_title('Net Proceeds by Holding Period', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nNote: Longer holding periods benefit from long-term capital gains treatment.")

## Exchange Fee Comparison

Different Indian crypto exchanges charge varying withdrawal fees. Use this tool to compare net proceeds across major platforms.

In [ ]:
# Compare major exchange fee structures
exchanges = [
    {'name': 'WazirX', 'fee_percent': 0.5, 'withdrawal_fee_usd': 2},
    {'name': 'CoinDCX', 'fee_percent': 0.1, 'withdrawal_fee_usd': 1},
    {'name': 'Crypto.com', 'fee_percent': 0.16, 'withdrawal_fee_usd': 3},
    {'name': 'Binance (P2P)', 'fee_percent': 0.001, 'withdrawal_fee_usd': 0},
]

exchange_results = []

for exch in exchanges:
    result = calculate_cashout(
        crypto_amount_usd=90000,
        sale_price_usd=1,
        cost_basis_usd=60000,
        holding_days=730,
        exchange_fee_percent=exch['fee_percent'],
        withdrawal_fee_usd=exch['withdrawal_fee_usd'],
        usd_to_inr_rate=83.5
    )
    exchange_results.append({
        'Exchange': exch['name'],
        'Fee %': exch['fee_percent'],
        'Withdrawal Fee (USD)': exch['withdrawal_fee_usd'],
        'Net INR': result['net_proceeds_inr']
    })

df_exchanges = pd.DataFrame(exchange_results)
df_exchanges = df_exchanges.sort_values('Net INR', ascending=False)
print("\n=== EXCHANGE COMPARISON (2 BTC, 2-year hold) ===")
print(df_exchanges.to_string(index=False))

# Chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
ax.barh(df_exchanges['Exchange'], df_exchanges['Net INR']/100000, color=colors)
ax.set_xlabel('Net INR (in hundreds of thousands)', fontsize=11)
ax.set_title('Net Proceeds by Exchange (2 BTC, 2-year holding)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Reference

[read this How to Cash Out Crypto in India: The Complete 2026 Withdrawal Guide article](https://protraderdaily.com/crypto/how-to-cash-out-crypto-in-india)
